# Advanced Problems: Python Truth Values and Boolean Contexts

This notebook contains advanced practice problems with complete solutions. The focus is on Python truth values, `__bool__`, `__len__`, numeric and container truthiness, defensive conditionals, and short-circuit behavior.

## Problem 1 — Truthiness Classification Matrix

Write `classify_truthiness(values)` returning `(repr(value), type_name, truth_value)` for each input.

In [1]:
from decimal import Decimal
from fractions import Fraction

def classify_truthiness(values):
    return [(repr(value), type(value).__name__, bool(value)) for value in values]

values = [0, 1, -3, 0.0, 0j, 2+0j, Decimal('0'), Decimal('0.01'),
          Fraction(0, 5), Fraction(2, 5), '', '0', [], [0], (), (False,),
          {}, {'x': None}, set(), {0}, None]

result = classify_truthiness(values)
result

[('0', 'int', False),
 ('1', 'int', True),
 ('-3', 'int', True),
 ('0.0', 'float', False),
 ('0j', 'complex', False),
 ('(2+0j)', 'complex', True),
 ("Decimal('0')", 'Decimal', False),
 ("Decimal('0.01')", 'Decimal', True),
 ('Fraction(0, 1)', 'Fraction', False),
 ('Fraction(2, 5)', 'Fraction', True),
 ("''", 'str', False),
 ("'0'", 'str', True),
 ('[]', 'list', False),
 ('[0]', 'list', True),
 ('()', 'tuple', False),
 ('(False,)', 'tuple', True),
 ('{}', 'dict', False),
 ("{'x': None}", 'dict', True),
 ('set()', 'set', False),
 ('{0}', 'set', True),
 ('None', 'NoneType', False)]

## Problem 2 — `__bool__` vs `__len__` Priority

Create classes showing that Python uses `__bool__` first, then `__len__`, then defaults to truthy.

In [2]:
class OnlyLen:
    def __init__(self, n):
        self.n = n
        self.log = []

    def __len__(self):
        self.log.append('__len__')
        return self.n


class OnlyBool:
    def __init__(self, flag):
        self.flag = flag
        self.log = []

    def __bool__(self):
        self.log.append('__bool__')
        return self.flag


class BoolAndLen:
    def __init__(self, flag, n):
        self.flag = flag
        self.n = n
        self.log = []

    def __bool__(self):
        self.log.append('__bool__')
        return self.flag

    def __len__(self):
        self.log.append('__len__')
        return self.n


examples = [OnlyLen(0), OnlyLen(3), OnlyBool(False), OnlyBool(True), BoolAndLen(False, 10)]
[(type(obj).__name__, bool(obj), obj.log) for obj in examples]

[('OnlyLen', False, ['__len__']),
 ('OnlyLen', True, ['__len__']),
 ('OnlyBool', False, ['__bool__']),
 ('OnlyBool', True, ['__bool__']),
 ('BoolAndLen', False, ['__bool__'])]

## Problem 3 — Invalid Truth-Value Protocols

`__bool__` must return a real bool. `__len__` must return a non-negative integer.

In [3]:
class BadBool:
    def __bool__(self):
        return 1

class BadLen:
    def __len__(self):
        return -1

def capture_exception(fn):
    try:
        fn()
    except Exception as ex:
        return type(ex).__name__, str(ex)

print(capture_exception(lambda: bool(BadBool())))
print(capture_exception(lambda: bool(BadLen())))

class GoodBool:
    def __init__(self, flag):
        self.flag = flag
    def __bool__(self):
        return bool(self.flag)

class GoodLen:
    def __init__(self, n):
        if n < 0:
            raise ValueError('length cannot be negative')
        self.n = n
    def __len__(self):
        return self.n

assert bool(GoodBool(1)) is True
assert bool(GoodBool(0)) is False
assert bool(GoodLen(0)) is False
assert bool(GoodLen(5)) is True
print('All tests passed.')

('TypeError', '__bool__ should return bool, returned int')
('ValueError', '__len__() should return >= 0')
All tests passed.


## Problem 4 — Do Not Confuse `None` with Empty Values

Write `first_or_default` and a generator-safe version preserving falsy first elements.

In [4]:
def first_or_default(items, default):
    if items:
        return items[0]
    return default

def first_or_default_sequence(items, default):
    if items is None:
        return default
    iterator = iter(items)
    try:
        return next(iterator)
    except StopIteration:
        return default

assert first_or_default([0], 'missing') == 0
assert first_or_default([False], 'missing') is False
assert first_or_default([''], 'missing') == ''
assert first_or_default([], 'missing') == 'missing'
assert first_or_default(None, 'missing') == 'missing'

assert first_or_default_sequence((x for x in [0]), 'missing') == 0
assert first_or_default_sequence((x for x in []), 'missing') == 'missing'
print('All tests passed.')

All tests passed.


## Problem 5 — Short-Circuit Evaluation and Safe Guards

Write `safe_first_upper(value)` using safe short-circuit checks.

In [5]:
def safe_first_upper(value):
    if value and hasattr(value[0], 'upper'):
        return value[0].upper()
    return None

assert safe_first_upper(None) is None
assert safe_first_upper('') is None
assert safe_first_upper([]) is None
assert safe_first_upper('abc') == 'A'
assert safe_first_upper(['python']) == 'PYTHON'
assert safe_first_upper([123]) is None
print('All tests passed.')

All tests passed.


## Problem 6 — The `or` Default Trap

`value = user_value or default` is wrong when falsy values like `0` are valid.

In [6]:
def normalize_limit(limit, default=100):
    if limit is None:
        return default
    if isinstance(limit, bool) or not isinstance(limit, int):
        raise ValueError('limit must be None or a non-negative integer')
    if limit < 0:
        raise ValueError('limit must be non-negative')
    return limit

assert normalize_limit(None) == 100
assert normalize_limit(0) == 0
assert normalize_limit(10) == 10

for bad in [False, True, -1, 3.5, '10', [], {}]:
    try:
        normalize_limit(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Expected ValueError for {bad!r}')

print('All tests passed.')

All tests passed.


## Problem 7 — Design a Truthy/Falsy Domain Object

Create `QueryResult`: truthy only if it has rows and no error.

In [7]:
class QueryResult:
    def __init__(self, rows, error=None):
        self.rows = list(rows)
        self.error = error

    def __len__(self):
        return len(self.rows)

    def __iter__(self):
        return iter(self.rows)

    def __bool__(self):
        return self.error is None and len(self.rows) > 0

    def __repr__(self):
        return f'QueryResult(rows={self.rows!r}, error={self.error!r})'

empty_ok = QueryResult([])
rows_ok = QueryResult([{'id': 1}, {'id': 2}])
rows_error = QueryResult([{'id': 1}], error='connection lost')

assert bool(empty_ok) is False
assert bool(rows_ok) is True
assert bool(rows_error) is False
assert len(rows_error) == 1
print(rows_ok)

QueryResult(rows=[{'id': 1}, {'id': 2}], error=None)


## Problem 8 — Boolean Operators Return Operands

`and` and `or` return operands, not necessarily bools. Then implement `coalesce_not_none`.

In [8]:
results = {
    "[] or 'fallback'": [] or 'fallback',
    "[1, 2] or 'fallback'": [1, 2] or 'fallback',
    "[] and 'next'": [] and 'next',
    "[1, 2] and 'next'": [1, 2] and 'next',
    "None or 0 or '' or 'done'": None or 0 or '' or 'done',
    "'first' and [] and 'last'": 'first' and [] and 'last',
}
results

{"[] or 'fallback'": 'fallback',
 "[1, 2] or 'fallback'": [1, 2],
 "[] and 'next'": [],
 "[1, 2] and 'next'": 'next',
 "None or 0 or '' or 'done'": 'done',
 "'first' and [] and 'last'": []}

In [9]:
def coalesce_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None

assert coalesce_not_none(None, 0, 5) == 0
assert coalesce_not_none(None, False, True) is False
assert coalesce_not_none(None, '', 'fallback') == ''
assert coalesce_not_none(None, [], [1]) == []
assert coalesce_not_none(None, None) is None
print('All tests passed.')

All tests passed.


## Problem 9 — Defensive Payload Validator

Validate an API payload while distinguishing missing, `None`, empty, and valid falsy values.

In [10]:
def validate_payload(payload):
    if not isinstance(payload, dict):
        raise ValueError('payload must be a dictionary')

    name = payload.get('name')
    if not isinstance(name, str):
        raise ValueError('name must be a string')

    cleaned_name = name.strip()
    if not cleaned_name:
        raise ValueError('name cannot be empty')

    raw_tags = payload.get('tags', None)
    if raw_tags is None:
        cleaned_tags = []
    else:
        if not isinstance(raw_tags, list):
            raise ValueError('tags must be a list of strings')
        cleaned_tags = []
        for tag in raw_tags:
            if not isinstance(tag, str):
                raise ValueError('each tag must be a string')
            cleaned_tag = tag.strip()
            if not cleaned_tag:
                raise ValueError('tags cannot contain empty strings')
            cleaned_tags.append(cleaned_tag)

    return {'name': cleaned_name, 'tags': cleaned_tags}

assert validate_payload({'name': ' Alice '}) == {'name': 'Alice', 'tags': []}
assert validate_payload({'name': 'Alice', 'tags': []}) == {'name': 'Alice', 'tags': []}
assert validate_payload({'name': 'Alice', 'tags': [' python ']}) == {'name': 'Alice', 'tags': ['python']}
print('All tests passed.')

All tests passed.


## Problem 10 — Lazy Boolean Evaluation

Create `LazyCheck`, which evaluates a function only when truthiness is requested and caches the result.

In [11]:
class LazyCheck:
    def __init__(self, fn):
        self.fn = fn
        self._evaluated = False
        self._value = None

    @property
    def evaluated(self):
        return self._evaluated

    def __bool__(self):
        if not self._evaluated:
            self._value = bool(self.fn())
            self._evaluated = True
        return self._value

calls = []

def expensive_true():
    calls.append('expensive_true called')
    return 'non-empty result'

check = LazyCheck(expensive_true)
assert check.evaluated is False
assert bool(check) is True
assert bool(check) is True
assert calls == ['expensive_true called']

calls_2 = []
lazy = LazyCheck(lambda: calls_2.append('ran') or True)
result = False and lazy

assert result is False
assert lazy.evaluated is False
assert calls_2 == []
print('All tests passed.')

All tests passed.


## Summary

- Use `if obj:` when Python's normal truth-value semantics are intended.
- Use `is None` when `None` has a distinct meaning from other falsy values.
- Avoid `x or default` when `0`, `False`, `''`, or `[]` are valid.
- Implement `__bool__` for domain-specific truthiness.
- Implement `__len__` for size, not business logic.
- Remember that `and` and `or` return operands.
- Order short-circuit checks from safest to riskiest.